<a href="https://colab.research.google.com/github/singhmishal2004-netizen/Stages_of_finetuning_on_youtube_transcript_data/blob/main/stages_of_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U peft bitsandbytes transformers accelerate

In [ ]:
!pip install -U trl

In [ ]:
!pip install PyMuPDF

In [ ]:
import fitz

In [ ]:
def extract_text_from_pdf(pdf_path):
    text_blocks = []
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text = page.get_text("text").strip()
            if text:
                text_blocks.append(text)
    return text_blocks

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving llm_finetuning_transcript.pdf to llm_finetuning_transcript (3).pdf


In [ ]:
import os
print(os.listdir("/content"))

['.config', 'tinyllama-lora', 'llm_finetuning_transcript (3).pdf', 'llm_finetuning_transcript.pdf', 'llm_finetuning_transcript (1).pdf', 'llm_finetuning_transcript (2).pdf', 'tinyllama-qlora', 'sample_data']


In [ ]:
pdf_texts = extract_text_from_pdf("/content/llm_finetuning_transcript.pdf")

In [ ]:
pdf_texts

["Page 1\nLLM Fine-Tuning — Transcript\nCleaned transcript for PDF extraction / fine-tuning practice\nHey. Hi everyone. Welcome back to my YouTube channel. My name is Sunonny and I'm back with another exciting\nand important video. So guys, from this video onwards, we're going to start the fine tuning of the large language\nmodel. As you know, I started one end to end LLM fine tuning playlist where I uploaded a 13th video so far. In the last\nvideo I discuss about the LLM quantization and apart from that I discuss each and every fundamental concept of the\nfinetuning. Now it's time to deep dive into the LLM fine tuning. Uh I will show you the practical with respect to each\nand every framework. So today I'm going to start with the hugging face ecosystem. In the next video you will find out\naxelottals then unslaught the llama factory and you will find out fine-tuning of different different model not only the text\nmodel and even uh I will fine-tune the multimodel which can handle the i

In [ ]:
import re

def chunk_text(pdf_texts, chunk_size=1000):
    chunks = []

    for page in pdf_texts:
        # Split on blank lines
        paragraphs = re.split(r'\n\s*\n+', page)

        current_chunk = ""

        for paragraph in paragraphs:
            paragraph = re.sub(r'\s+', ' ', paragraph).strip()

            if not paragraph:
                continue

            # Add paragraph to current chunk
            if len(current_chunk) + len(paragraph) <= chunk_size:
                current_chunk += " " + paragraph
            else:
                if current_chunk:
                    chunks.append(current_chunk.strip())

                current_chunk = paragraph

        # Add remaining text
        if current_chunk:
            chunks.append(current_chunk.strip())

    return chunks

In [ ]:
chunks = chunk_text(pdf_texts, chunk_size=1000)

print("Number of chunks:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0])

Number of chunks: 71

First chunk:

Page 1 LLM Fine-Tuning — Transcript Cleaned transcript for PDF extraction / fine-tuning practice Hey. Hi everyone. Welcome back to my YouTube channel. My name is Sunonny and I'm back with another exciting and important video. So guys, from this video onwards, we're going to start the fine tuning of the large language model. As you know, I started one end to end LLM fine tuning playlist where I uploaded a 13th video so far. In the last video I discuss about the LLM quantization and apart from that I discuss each and every fundamental concept of the finetuning. Now it's time to deep dive into the LLM fine tuning. Uh I will show you the practical with respect to each and every framework. So today I'm going to start with the hugging face ecosystem. In the next video you will find out axelottals then unslaught the llama factory and you will find out fine-tuning of different different model not only the text model and even uh I will fine-tune the multimode

In [ ]:
data = [{"text": i} for i in chunks]

In [ ]:
from datasets import Dataset
dataset = Dataset.from_list(data)

In [ ]:
dataset

Dataset({
    features: ['text'],
    num_rows: 71
})

In [ ]:
dataset[0]

{'text': "Page 1 LLM Fine-Tuning — Transcript Cleaned transcript for PDF extraction / fine-tuning practice Hey. Hi everyone. Welcome back to my YouTube channel. My name is Sunonny and I'm back with another exciting and important video. So guys, from this video onwards, we're going to start the fine tuning of the large language model. As you know, I started one end to end LLM fine tuning playlist where I uploaded a 13th video so far. In the last video I discuss about the LLM quantization and apart from that I discuss each and every fundamental concept of the finetuning. Now it's time to deep dive into the LLM fine tuning. Uh I will show you the practical with respect to each and every framework. So today I'm going to start with the hugging face ecosystem. In the next video you will find out axelottals then unslaught the llama factory and you will find out fine-tuning of different different model not only the text model and even uh I will fine-tune the multimodel which can handle the ima

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def tokenize_fn(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

In [ ]:
tokenized = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"]
)

Map:   0%|          | 0/71 [00:00<?, ? examples/s]

In [ ]:
tokenized

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 71
})

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="./tinyllama-lora",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized
)

trainer.train()

OutOfMemoryError: CUDA out of memory. Tried to allocate 44.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 43.81 MiB is free. Including non-PyTorch memory, this process has 14.52 GiB memory in use. Of the allocated memory 14.24 GiB is allocated by PyTorch, and 148.89 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:

# # -------------------------------------------------------------
# # 1️⃣ Install dependencies
# # -------------------------------------------------------------
# !pip install -q transformers datasets accelerate bitsandbytes

# # -------------------------------------------------------------
# # 2️⃣ Imports
# # -------------------------------------------------------------
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
# from datasets import load_dataset
# import os, re

# # -------------------------------------------------------------
# # 3️⃣ Load base model and tokenizer
# # -------------------------------------------------------------
# model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

# # -------------------------------------------------------------
# # 4️⃣ Freeze everything
# # -------------------------------------------------------------
# for param in model.parameters():
#     param.requires_grad = False

# # -------------------------------------------------------------
# # 5️⃣ Unfreeze last 4 transformer blocks + lm_head
# # -------------------------------------------------------------
# for name, param in model.named_parameters():
#     if any(f"layers.{i}." in name for i in range(20, 24)):  # last 4 layers
#         param.requires_grad = True
#     if "lm_head" in name:
#         param.requires_grad = True

# # Verify how many parameters are trainable
# trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
# total = sum(p.numel() for p in model.parameters())
# print(f"✅ Trainable: {trainable/total*100:.2f}% of total parameters")

# # -------------------------------------------------------------
# # 6️⃣ Load your dataset
# # -------------------------------------------------------------
# dataset = load_dataset("json", data_files={"train": "pharma_non_instruction.jsonl"})

# def tokenize_fn(examples):
#     tokens = tokenizer(
#         examples["text"],
#         truncation=True,
#         padding="max_length",
#         max_length=512
#     )
#     tokens["labels"] = tokens["input_ids"].copy()
#     return tokens

# tokenized = dataset["train"].map(tokenize_fn, batched=True, remove_columns=["text"])

# # -------------------------------------------------------------
# # 7️⃣ Data collator for causal LM
# # -------------------------------------------------------------
# data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# # -------------------------------------------------------------
# # 8️⃣ Training arguments
# # -------------------------------------------------------------
# os.environ["WANDB_DISABLED"] = "true"  # disable wandb
# training_args = TrainingArguments(
#     output_dir="./tinyllama-pharma-last4",
#     overwrite_output_dir=True,
#     num_train_epochs=2,
#     per_device_train_batch_size=2,
#     gradient_accumulation_steps=4,
#     learning_rate=1e-4,        # slightly higher since fewer params
#     fp16=True,
#     logging_steps=20,
#     save_steps=200,
#     save_total_limit=2,
#     report_to="none"
# )

# # -------------------------------------------------------------
# # 9️⃣ Trainer setup
# # -------------------------------------------------------------
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized,
#     data_collator=data_collator
# )

# # -------------------------------------------------------------
# # 🔟 Start training
# # -------------------------------------------------------------
# trainer.train()

# # -------------------------------------------------------------
# # 11️⃣ Save final model
# # -------------------------------------------------------------
# trainer.save_model("./tinyllama-pharma-last4-final")
# print("\n✅ Training completed and model saved!")


In [ ]:
del model

import gc
import torch

gc.collect()
torch.cuda.empty_cache()

In [ ]:
print(torch.cuda.memory_allocated() / 1024**3, "GB allocated")
print(torch.cuda.memory_reserved() / 1024**3, "GB reserved")

0.015869140625 GB allocated
0.01953125 GB reserved


In [ ]:
!nvidia-smi

Wed Oct 28 22:59:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   64C    P0             31W /   70W |     163MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -U peft bitsandbytes transformers accelerate

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [ ]:
model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model)

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def tokenize_fn(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens


In [ ]:
tokenized = dataset.map(tokenize_fn, batched=True)

Map:   0%|          | 0/71 [00:00<?, ? examples/s]

In [ ]:
tokenized

Dataset({
    features: ['text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 71
})

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    model,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
from peft import prepare_model_for_kbit_training

base_model = prepare_model_for_kbit_training(base_model)
base_model.config.use_cache = False

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)

qlora_model = get_peft_model(base_model, lora_config)

qlora_model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [ ]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="./tinyllama-qlora",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=10,
    save_total_limit=1,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=qlora_model,
    args=args,
    train_dataset=tokenized
)

trainer.train()

Step,Training Loss
10,3.315095
20,3.233917


TrainOutput(global_step=27, training_loss=3.251284917195638, metrics={'train_runtime': 76.0426, 'train_samples_per_second': 2.801, 'train_steps_per_second': 0.355, 'total_flos': 338827869683712.0, 'train_loss': 3.251284917195638, 'epoch': 3.0})

In [ ]:
import torch

print("Allocated:", torch.cuda.memory_allocated() / 1024**3, "GB")
print("Reserved:", torch.cuda.memory_reserved() / 1024**3, "GB")
print("Free:", (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1024**3, "GB")

Allocated: 0.997103214263916 GB
Reserved: 1.326171875 GB
Free: 13.566068172454834 GB


In [ ]:
print(type(qlora_model))

<class 'peft.peft_model.PeftModelForCausalLM'>


In [ ]:
qlora_model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [ ]:
prompt = "What is fine-tuning and why is it used for large language models?"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = qlora_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


What is fine-tuning and why is it used for large language models?
* 19:19 What is a BERT model and what are its benefits?
* 19:35 What are the pros and cons of using BERT?
* 20:32 What are the pros and cons of using BERT for the language model?
* 22:11 What is the difference between BERT and LM?
* 23:15 What are the pros and cons of using LM?


In [ ]:
base_model.eval()

with torch.no_grad():
    base_outputs = base_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(base_outputs[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


What is fine-tuning and why is it used for large language models?
What is fine-tuning and why is it used for large language models?
Fine-tuning is a process of training a model to learn from a large dataset.
The model is trained on a large dataset, and then it is fine-tuned on a smaller dataset.
Fine-tuning is a process of training a model to learn from a large dataset.
The model is trained on a large dataset, and then it is fine-tuned


In [ ]:
qlora_model.eval()

with torch.no_grad():
    lora_outputs = qlora_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(lora_outputs[0], skip_special_tokens=True))